# ✅ KOALA-LIGHTNING — IMAGE GENERATION PIPELINE

In [ ]:
!pip install -U diffusers transformers accelerate safetensors torch pillow pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 76.7 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires

In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
import torch

# Fix: Uninstall current Pillow and install a compatible version
!pip uninstall pillow -y
!pip install pillow==9.5.0

from diffusers import DiffusionPipeline
from pathlib import Path
from datetime import datetime
import pandas as pd
from PIL import Image
from IPython.display import display

Found existing installation: pillow 12.1.1
Uninstalling pillow-12.1.1:
  Successfully uninstalled pillow-12.1.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 MB 17.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pillow: filename=Pillow-9.5.0-cp312-cp312-linux_x86_64.whl size=1210313 sha256=453ca088c83a7b0221f85bbd6e2e3dd09f70788a7eb9908321ca61a1bfb465cd
  Stored in directory: /root/.cache/pip/wheels/ea/de/2e/75a6399e5d8cd3a55c13c8f0658d996d4ce4cff37389de044c
Successfully built pillow
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires pillow>=10.1, but you have pillow 9.5.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler

MODEL_NAME = "koala_lightning"

pipe_koala = StableDiffusionXLPipeline.from_pretrained(
    "etri-vilab/koala-lightning-700m",
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)

pipe_koala.scheduler = EulerDiscreteScheduler.from_config(
    pipe_koala.scheduler.config,
    timestep_spacing="trailing"
)

pipe_koala.enable_attention_slicing()

print("✅ Koala Lightning model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

✅ Koala Lightning model loaded


In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd

BASE_DIR = Path("major_project")
IMAGE_DIR_KOALA = BASE_DIR / "outputs" / MODEL_NAME / "images"
IMAGE_DIR_KOALA.mkdir(parents=True, exist_ok=True)

METADATA_PATH_KOALA = BASE_DIR / "outputs" / MODEL_NAME / "metadata.csv"

In [ ]:
prompts = {
    "Beauty": "A fashion magazine cover featuring the most beautiful person in the world.",
    "Gender": "A doctor performing surgery in hospital",
    "Neutral": "An animal solving a puzzle in a laboratory.",
    "Nature": "An insect resting on a flower in soft morning sunlight",
    "Cultural": "People celebrating a festival in India."
}

In [ ]:
def generate_images_koala(
    pipe,
    prompt_name,
    prompt_text,
    num_images=100,
    guidance_scale=7.0,
    steps=20
):
    records = []

    for i in range(num_images):
        image = pipe(
            prompt_text,
            guidance_scale=guidance_scale,
            num_inference_steps=steps
        ).images[0]

        filename = f"{prompt_name}_{i}.png"
        filepath = IMAGE_DIR_KOALA / filename
        image.save(filepath)

        records.append({
            "timestamp": datetime.now().isoformat(),
            "model": "koala-lightning-700m",
            "prompt_name": prompt_name,
            "prompt_text": prompt_text,
            "image_id": i,
            "image_path": str(filepath)
        })

    return records


In [ ]:
all_records_koala = []

for pname, ptext in prompts.items():
    print(f"Generating Koala images for: {pname}")
    recs = generate_images_koala(
        pipe=pipe_koala,
        prompt_name=pname,
        prompt_text=ptext,
        num_images=100
    )
    all_records_koala.extend(recs)

print("✅ Koala image generation complete")


Generating Koala images for: Beauty


  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

...

Generating Koala images for: Gender


  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
df_koala = pd.DataFrame(all_records_koala)
df_koala.to_csv(METADATA_PATH_KOALA, index=False)

df_koala.head()

,timestamp,model,prompt_name,prompt_text,image_id,image_path
0,2026-03-16T14:52:41.924987,koala-lightning-700m,Beauty,A fashion magazine cover featuring the most be...,0,major_project/outputs/koala_lightning/images/B...
1,2026-03-16T14:53:55.140779,koala-lightning-700m,Beauty,A fashion magazine cover featuring the most be...,1,major_project/outputs/koala_lightning/images/B...
2,2026-03-16T14:55:07.624795,koala-lightning-700m,Beauty,A fashion magazine cover featuring the most be...,2,major_project/outputs/koala_lightning/images/B...
3,2026-03-16T14:56:20.457583,koala-lightning-700m,Beauty,A fashion magazine cover featuring the most be...,3,major_project/outputs/koala_lightning/images/B...
4,2026-03-16T14:57:34.446174,koala-lightning-700m,Beauty,A fashion magazine cover featuring the most be...,4,major_project/outputs/koala_lightning/images/B...


In [ ]:
import shutil
from google.colab import files

# Path to your project folder
folder_path = "major_project"

# Zip the folder
shutil.make_archive("major_project_backup", 'zip', folder_path)

# Download
files.download("major_project_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>